# 02 — Reading & Writing Data

**What you will learn:**
- Reading CSV, JSON, Parquet, and Delta formats
- Common reader options (header, inferSchema, delimiter, schema)
- Writing DataFrames with different save modes
- Writing to Delta Lake (managed and external tables)
- Checking partition files written to storage

## Setup — Create Sample Files in DBFS for Practice

In [ ]:
import json

# Write a sample CSV to DBFS
csv_data = """emp_id,name,department,salary,hire_date
1,Alice,Engineering,95000,2022-01-15
2,Bob,Marketing,72000,2021-06-01
3,Charlie,Engineering,105000,2020-03-20
4,Diana,HR,68000,2023-09-10
5,Eve,Marketing,78000,2022-11-05
6,Frank,Engineering,88000,2021-04-12
7,Grace,HR,71000,2022-07-30
"""

dbutils.fs.put("/tmp/practice/employees.csv", csv_data, overwrite=True)

# Write sample JSON (newline-delimited JSON = JSON Lines)
json_data = "\n".join([
    json.dumps({"order_id": 1001, "customer": "Alice", "amount": 250.50, "status": "COMPLETED"}),
    json.dumps({"order_id": 1002, "customer": "Bob",   "amount": 89.99,  "status": "PENDING"}),
    json.dumps({"order_id": 1003, "customer": "Alice", "amount": 430.00, "status": "COMPLETED"}),
    json.dumps({"order_id": 1004, "customer": "Diana", "amount": 120.75, "status": "CANCELLED"}),
])
dbutils.fs.put("/tmp/practice/orders.json", json_data, overwrite=True)

print("Sample files created.")

## 1. Reading CSV

In [ ]:
# --- Basic CSV read with options ---
df_csv = (
    spark.read
    .option("header", "true")          # first row = column names
    .option("inferSchema", "true")     # auto-detect data types (slow on large files)
    .option("delimiter", ",")          # default is comma
    .option("nullValue", "NA")         # treat "NA" string as null
    .option("emptyValue", "")          # treat empty string as empty (not null)
    .csv("/tmp/practice/employees.csv")
)

df_csv.printSchema()
df_csv.show()

In [ ]:
# --- Better: define schema explicitly (faster, no scan) ---
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

emp_schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("name",       StringType(),  True),
    StructField("department", StringType(),  True),
    StructField("salary",     DoubleType(),  True),
    StructField("hire_date",  StringType(),  True),
])

df_emp = (
    spark.read
    .option("header", "true")
    .schema(emp_schema)
    .csv("/tmp/practice/employees.csv")
)

df_emp.printSchema()
df_emp.show()

## 2. Reading JSON (Newline-Delimited)

In [ ]:
df_json = (
    spark.read
    .option("multiLine", "false")   # false = each line is one JSON object (JSON Lines)
    .json("/tmp/practice/orders.json")
)

df_json.printSchema()
df_json.show()

## 3. Writing Data — Save Modes

| Mode | Behavior |
|---|---|
| `overwrite` | Delete existing data and write new |
| `append`    | Add new data alongside existing data |
| `ignore`    | Do nothing if path already exists |
| `error` (default) | Throw error if path already exists |

In [ ]:
# Write to Parquet (columnar, compressed — preferred format for data lakes)
(
    df_emp
    .write
    .mode("overwrite")
    .parquet("/tmp/practice/employees_parquet/")
)

print("Written to Parquet.")

In [ ]:
# Write to CSV
(
    df_emp
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv("/tmp/practice/employees_out_csv/")
)

In [ ]:
# Write to JSON
(
    df_emp
    .write
    .mode("overwrite")
    .json("/tmp/practice/employees_out_json/")
)

## 4. Reading Parquet

In [ ]:
df_parquet = spark.read.parquet("/tmp/practice/employees_parquet/")
df_parquet.printSchema()
df_parquet.show()

## 5. Writing & Reading Delta Lake

In [ ]:
# Write as Delta
(
    df_emp
    .write
    .format("delta")
    .mode("overwrite")
    .save("/tmp/practice/employees_delta/")
)

print("Written as Delta.")

In [ ]:
# Read Delta
df_delta = spark.read.format("delta").load("/tmp/practice/employees_delta/")
df_delta.show()

## 6. Partitioned Writes

Partitioning organizes files into sub-folders by column values.
Queries that filter on the partition column skip irrelevant files (partition pruning).

In [ ]:
(
    df_emp
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("department")          # one sub-folder per department
    .save("/tmp/practice/employees_partitioned/")
)

# List the partition folders
display(dbutils.fs.ls("/tmp/practice/employees_partitioned/"))

In [ ]:
# Read with partition filter — Spark reads only the Engineering folder
df_eng = (
    spark.read
    .format("delta")
    .load("/tmp/practice/employees_partitioned/")
    .filter("department = 'Engineering'")
)
df_eng.show()

## 7. Creating Delta Tables in Unity Catalog / Metastore

Writing to a path creates a Delta file. Registering it as a table
lets you query it with SQL using `catalog.schema.table` syntax.

In [ ]:
# Managed table — Databricks controls the storage location
df_emp.write.format("delta").mode("overwrite").saveAsTable("default.employees")

# Read it back via SQL
spark.sql("SELECT * FROM default.employees").show()

In [ ]:
# External table — you control the storage path
(
    df_emp.write
    .format("delta")
    .mode("overwrite")
    .option("path", "/tmp/practice/ext_employees/")
    .saveAsTable("default.ext_employees")
)

spark.sql("SELECT * FROM default.ext_employees").show()

## 8. Reading Multiple Files at Once

In [ ]:
# Read all CSVs in a folder using wildcard
# spark.read.csv("/mnt/datalake/raw/2024/*/orders_*.csv")

# Read a list of specific files
# spark.read.csv(["/path/file1.csv", "/path/file2.csv"])

# Add source file name as a column (useful for auditing)
from pyspark.sql.functions import input_file_name

df_with_source = (
    spark.read
    .option("header", "true")
    .csv("/tmp/practice/employees.csv")
    .withColumn("source_file", input_file_name())
)
df_with_source.show(truncate=False)

## Key Takeaways

| Operation | Code Pattern |
|---|---|
| Read CSV | `spark.read.option("header","true").schema(s).csv(path)` |
| Read JSON | `spark.read.json(path)` |
| Read Parquet | `spark.read.parquet(path)` |
| Read Delta | `spark.read.format("delta").load(path)` |
| Write overwrite | `.write.mode("overwrite").parquet(path)` |
| Write partitioned | `.write.partitionBy("col").save(path)` |
| Managed table | `.write.saveAsTable("schema.table")` |
| External table | `.write.option("path", p).saveAsTable("schema.table")` |
| Always prefer explicit schema | Avoids full-scan inference, enforces types |